In [1]:
import sys
sys.path.append('../')

# NaHCO3-CO2(g) equilibrium
- Calculate pH of 0.01 mol/L NaHCO3 aq. before and after contact with 400 ppm CO2(g).
- Considers activity coefficient (Davies model)

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

## Settings

In [3]:
import eq_solver
import summarize

In [4]:
temp_deg = 25.0
c = 0.01
s_gas = eq_solver.EqSystem.from_yaml('./Na-CO3-gas.yaml')
s_nogas = eq_solver.EqSystem.from_yaml('./Na-CO3-nogas.yaml')
cond_gas = eq_solver.Conditions.from_dict(s_gas, {'carbonate': 400e-6, 'sodium': c})
cond_nogas = eq_solver.Conditions.from_dict(s_nogas, {'carbonate': c, 'sodium': c})

## Calculation

In [5]:
f_gas = s_gas.make_f(cond_gas)
r_gas = f_gas.solve()
f_nogas = s_nogas.make_f(cond_nogas)
r_nogas = f_nogas.solve()
print(r_gas.rmse, r_nogas.rmse)

2.198129442157285e-15 4.796711693331816e-16


In [6]:
from pprint import pprint
print('------ f_nogas info -----')
pprint(f_nogas)
print('------ f_gas info -----')
pprint(f_gas)

------ f_nogas info -----
FitFunc(system=EqSystem(activity_model='davies',
                        cpt_names=array(['proton', 'carbonate', 'sodium'], dtype='<U9'),
                        cpt_base=array(['H+', 'CO2aq', 'Na+'], dtype='<U5'),
                        cpt_cstr=array([0, 1, 2], dtype=uint8),
                        cpt_charge=array([1, 0, 1]),
                        spc_names=array(['H+', 'OH-', 'Na+', 'CO2aq', 'HCO3-', 'CO3^2-'], dtype='<U6'),
                        spc_phase=array([1, 1, 1, 1, 1, 1], dtype=uint8),
                        spc_logK=array([  0.  , -14.  ,   0.  ,  -0.  ,  -6.3 , -16.63]),
                        composition_matrix=array([[ 1.,  0., -0.],
       [-1., -0., -0.],
       [ 0., -0.,  1.],
       [-0.,  1.,  0.],
       [-1.,  1.,  0.],
       [-2.,  1.,  0.]]),
                        temperature_K=298.15,
                        has_charge_cstr=True,
                        n_loga_free=np.int64(2),
                        n_phase_amount=np.in

In [7]:
print(f'pH before contact with gas: {r_nogas.pH():.2f}')
print(f'pH after contact with 400 ppm CO2: {r_gas.pH():.2f}')

pH before contact with gas: 8.22
pH after contact with 400 ppm CO2: 9.06


In [8]:
# comparison of carbonate species distribution
print('------ concentrations (aq.) ------')
df = pd.DataFrame([r_nogas.distribution('carbonate'), r_gas.distribution('carbonate')], index=['no gas', 'with gas'])
# mole of CO2(g) is meaningless because we can't determine the amount (volume) of gas phase
del df['CO2(g)']
df

------ concentrations (aq.) ------


,CO2aq,HCO3-,CO3^2-
no gas,0.000106,0.00979,0.000104
with gas,0.000014,0.00870,0.000644
